# Kiwi 형태소 분석 입문

**환경:** Google Colab / Jupyter Notebook
**준비물:** 없음 (엑셀 파일 없이 예시 문장으로 실습)


### 실습 진행 과정
| 단계 | 내용 | 확인할 결과 |
|---|---|---|
| 0 | Kiwi 설치·불러오기 | 오류 없이 `kiwi` 생성 |
| 1 | 형태소 분석 `tokenize()` | 문장이 형태소와 품사로 나뉨 |
| 2 | 품사표 읽기 | `NNG`, `VA` 등 태그 의미 |
| 3 | 명사만 뽑기 | 핵심 단어 목록 |
| 4 | 문장 나누기 `split_into_sents()` | 마침표 없어도 문장 분리 |
| 5 | 띄어쓰기 교정 `space()` | 붙어 쓴 문장 교정 |
| 6 | 사용자 사전 `add_user_word()` | 잘못 나뉜 단어 바로잡기 |
| 7 | 불용어 `Stopwords` | 의미 없는 형태소 제거 |
| 8 | 여러 문장 한 번에 분석 + 단어 세기 | 자주 나온 명사 순위 |
| 9 | 연습 문제 | 직접 코드 작성 |

> 위에서부터 셀을 **순서대로** 실행하세요. `Shift + Enter`로 실행합니다.

## 0. Kiwi 설치·불러오기
Kiwi는 파이썬용 한국어 형태소 분석기입니다. 처음 한 번만 설치하면 됩니다.

In [1]:
# Kiwi 설치 (처음 한 번만)
!pip install -q kiwipiepy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 8.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.4 MB/s eta 0:00:00


In [2]:
# Kiwi 불러오기
from kiwipiepy import Kiwi

# 형태소 분석기 만들기
kiwi = Kiwi()

## 1. 형태소 분석하기
**형태소**는 뜻을 가진 가장 작은 말의 단위입니다. `tokenize()`에 문장을 넣으면 형태소 목록이 나옵니다.

In [3]:
# 분석할 문장
text = "배송이 빨라서 좋고 가격도 저렴해서 만족합니다"

# 형태소 분석 결과 보기
kiwi.tokenize(text)

[Token(form='배송', tag='NNG', start=0, len=2),
 Token(form='이', tag='JKS', start=2, len=1),
 Token(form='빠르', tag='VA', start=4, len=1),
 Token(form='어서', tag='EC', start=5, len=2),
 Token(form='좋', tag='VA', start=8, len=1),
 Token(form='고', tag='EC', start=9, len=1),
 Token(form='가격', tag='NNG', start=11, len=2),
 Token(form='도', tag='JX', start=13, len=1),
 Token(form='저렴하', tag='VA', start=15, len=3),
 Token(form='어서', tag='EC', start=17, len=2),
 Token(form='만족', tag='NNG', start=20, len=2),
 Token(form='하', tag='XSA', start=22, len=1),
 Token(form='ᆸ니다', tag='EF', start=22, len=3)]

결과의 `form`은 **형태소**, `tag`는 **품사**입니다. 둘만 꺼내 보겠습니다.

In [4]:
# 형태소와 품사만 한 줄씩 출력
for token in kiwi.tokenize(text):
    print(token.form, token.tag)

배송 NNG
이 JKS
빠르 VA
어서 EC
좋 VA
고 EC
가격 NNG
도 JX
저렴하 VA
어서 EC
만족 NNG
하 XSA
ᆸ니다 EF


## 2. 품사표 읽기
자주 쓰는 품사만 알아 두면 됩니다.

| 태그 | 뜻 | 예 |
|---|---|---|
| `NNG` | 일반 명사 | 배송, 가격 |
| `NNP` | 고유 명사 | 서울, 삼성 |
| `VV` | 동사 | 먹(다), 사(다) |
| `VA` | 형용사 | 좋(다), 빠르(다) |
| `MAG` | 부사 | 정말, 너무 |
| `J`로 시작 | 조사 | 이, 가, 를 |
| `E`로 시작 | 어미 | 어요, 습니다 |

동사·형용사는 `좋`, `빠르`처럼 **어간**만 나옵니다. `다`를 붙여 읽으면 됩니다.

## 3. 명사만 뽑기
리뷰에서 **무엇에 대한 이야기인지** 알고 싶을 때 명사만 뽑습니다.

In [5]:
# 품사가 NNG(일반 명사)인 형태소만 골라내기
[t.form for t in kiwi.tokenize(text) if t.tag == "NNG"]

['배송', '가격', '만족']

## 4. 문장 나누기
마침표가 없는 리뷰도 문장 단위로 나눌 수 있습니다.

In [6]:
# 마침표 없이 이어 쓴 리뷰
review = "배송이 빨라요 포장도 꼼꼼했어요 다음에 또 살게요"

# 문장 단위로 나누어 출력
for sent in kiwi.split_into_sents(review):
    print(sent.text)

배송이 빨라요
포장도 꼼꼼했어요
다음에 또 살게요


## 5. 띄어쓰기 교정
띄어쓰기 없이 쓴 문장을 고쳐 줍니다.

In [7]:
# 붙여 쓴 문장의 띄어쓰기 교정
kiwi.space("배송이너무늦어서불편했어요")

'배송이 너무 늦어서 불편했어요'

## 6. 사용자 사전
분석기가 모르는 단어는 잘못 나뉠 수 있습니다. 그럴 때 단어를 직접 등록합니다.

In [8]:
# 등록 전: '쿨링감'이 여러 조각으로 나뉨
print([t.form for t in kiwi.tokenize("쿨링감이 좋아요")])

# '쿨링감'을 일반 명사(NNG)로 등록
kiwi.add_user_word("쿨링감", "NNG")

# 등록 후: 한 단어로 분석됨
print([t.form for t in kiwi.tokenize("쿨링감이 좋아요")])

['쿨', '링', '감', '이', '좋', '어요']
['쿨링감', '이', '좋', '어요']


## 7. 불용어 제거
**불용어**는 자주 나오지만 의미가 적은 말(조사, 어미 등)입니다. Kiwi에 기본 불용어 목록이 들어 있습니다.

In [9]:
# 기본 불용어 목록 불러오기
from kiwipiepy.utils import Stopwords
stopwords = Stopwords()

# 불용어를 빼고 분석
[t.form for t in kiwi.tokenize(text, stopwords=stopwords)]

['배송', '빠르', '좋', '가격', '저렴하', '만족', 'ᆸ니다']

## 8. 여러 문장 한 번에 분석하고 단어 세기
문장 목록을 통째로 넣으면 한 번에 분석합니다. 그 결과에서 명사를 세어 봅니다.

In [10]:
# 리뷰 5개
reviews = [
    "배송이 빠르고 포장이 꼼꼼해요",
    "가격 대비 품질이 좋아요",
    "배송은 느렸지만 품질은 만족합니다",
    "포장이 찢어져서 왔어요",
    "가격이 저렴하고 배송도 빨라요",
]

# 단어 세기 도구 불러오기
from collections import Counter

# 모든 리뷰의 명사 모으기
nouns = [t.form for tokens in kiwi.tokenize(reviews) for t in tokens if t.tag == "NNG"]

# 많이 나온 명사 3개
Counter(nouns).most_common(3)

[('배송', 3), ('포장', 2), ('가격', 2)]

## 9. 연습 문제
아래 빈 셀에 직접 코드를 써 보세요. 막히면 위 예제를 참고합니다.

**문제 1.** `"향이 은은하고 발림성이 좋아요"`에서 형태소와 품사를 출력하세요.
**문제 2.** 같은 문장에서 형용사(`VA`)만 뽑으세요.
**문제 3.** `"발림성"`이 어떻게 나뉘는지 확인하고, 사용자 사전에 등록한 뒤 다시 분석하세요.

In [11]:
# 문제 1: 여기에 코드 작성

In [12]:
# 문제 2: 여기에 코드 작성

In [13]:
# 문제 3: 여기에 코드 작성

<details>
<summary><b>정답 보기</b> (클릭)</summary>

```python
# 문제 1: 형태소와 품사 출력
for t in kiwi.tokenize("향이 은은하고 발림성이 좋아요"):
    print(t.form, t.tag)

# 문제 2: 형용사만 뽑기
[t.form for t in kiwi.tokenize("향이 은은하고 발림성이 좋아요") if t.tag == "VA"]

# 문제 3: 사용자 사전 등록 전후 비교
print([t.form for t in kiwi.tokenize("발림성이 좋아요")])
kiwi.add_user_word("발림성", "NNG")
print([t.form for t in kiwi.tokenize("발림성이 좋아요")])
```
문제 2에서 `은은하`가 나오지 않는 것을 확인해 보세요. `은은하다`는 `은은(XR, 어근) + 하(XSA)`로 나뉩니다. 이런 경우는 심화 실습에서 다룹니다.
</details>

### 정리
- `kiwi.tokenize(문장)` → 형태소(`form`)와 품사(`tag`)
- `split_into_sents()` 문장 분리, `space()` 띄어쓰기 교정
- `add_user_word()` 사용자 사전, `Stopwords` 불용어
- 문장 **목록**을 넣으면 여러 문장을 한 번에 분석

**참고:** [Kiwi 공식 저장소(kiwipiepy)](https://github.com/bab2min/kiwipiepy)